In [1]:
import jax
import jax.numpy as jnp
from flax import nnx


class MyModule(nnx.Module):
    constant: jax.Array
    constant_variable: nnx.Variable
    w: nnx.Param
    deteinistic: bool = False

    def __init__(self):
        self.deterministic = False
        self.constant = jnp.array(42.0)  # Immutable constant
        self.constant_variable = nnx.Variable(self.constant)
        self.w = nnx.Param(jnp.array(1.0))  # Trainable param

    def __call__(self, x):
        self.constant_variable += 1
        return x * self.w.value * self.constant


model = MyModule()
model(jnp.array(1.0))


def loss_fn(model, x):
    y = model(x)
    return jnp.mean(y**2)


model.train()
loss, grads = nnx.value_and_grad(loss_fn)(model, x=jnp.array([1.0]))

nnx.display(nnx.state(model))

In [2]:
import jax
import jax.numpy as jnp

# Assume f is a function f(x, y) that returns a vector
# x is a vector (dim n)
# y is a vector (dim m)
# w is a vector (dim p, same as f output dimension)
# v is a vector (dim m)


# Example function f: R^n x R^m -> R^p
# Replace this with your actual function
def f(x, y):
    """
    Example function: f(x, y) = [sum(x)*sum(y), sum(x)*y[0], x[0]*sum(y), ...]
    Input: x (n,), y (m,)
    Output: vector (p,)
    """
    n_dim = x.shape[0]
    m_dim = y.shape[0]
    p_dim = 10  # Example output dimension

    # Ensure dimensions are compatible for a realistic example
    # (Your actual f might have different logic/dependencies)
    assert n_dim == x_val.shape[0]
    assert m_dim == y_val.shape[0]
    assert p_dim == w_val.shape[0]  # Output dim must match w

    # A simple example computation involving elements from both x and y
    # f_i(x, y) = (sum(x) + i) * (sum(y) + i) + x[i % n] * y[i % m]
    result = jnp.zeros(p_dim)
    sum_x = jnp.sum(x)
    sum_y = jnp.sum(y)
    for i in range(p_dim):
        result = result.at[i].set((sum_x + i + 1) * (sum_y + i + 1) + x[i % n_dim] * y[i % m_dim])

    return result


# --- Define input dimensions (example) ---
n = 5  # Dimension of x (n is small)
m = 100  # Dimension of y (m is large)
p = 10  # Dimension of the output of f (and w)

# --- Create some dummy data ---
key = jax.random.PRNGKey(0)
x_val = jax.random.normal(key, (n,))
y_val = jax.random.normal(key, (m,))
w_val = jax.random.normal(key, (p,))
v_val = jax.random.normal(key, (m,))

# --- Implementation of w.T @ (d^2f/dxdy) @ v ---


# Step 1: Compute the JVP of f with respect to y in the direction v
# We need a function that takes y and returns f(x_val, y) for a fixed x_val
def f_of_y_at_x_val(y_arg):
    return f(x_val, y_arg)


# Compute the JVP df/dy @ v. jax.jvp returns (primal, tangent). We need the tangent.
# jax.jvp(function, (primals,), (tangents,))
# Here, function is f_of_y_at_x_val, primal is y_val, tangent is v_val
df_dy_v = jax.jvp(f_of_y_at_x_val, (y_val,), (v_val,))[1]
# df_dy_v is a p-dimensional vector


# Step 2: Define a scalar function of x: g(x) = w^T @ (df(x, y_val)/dy @ v_val)
# We need a function that takes x and returns w_val^T @ (df(x, y_val)/dy @ v_val)
def scalar_fn_of_x(x_arg):
    # Inside this function, y_val and v_val are fixed closures
    # We need the JVP of f with respect to y, but now f depends on x_arg
    # So, we need a function of y specifically for this x_arg
    def f_of_y_at_x_arg(y_arg):
        return f(x_arg, y_arg)

    # Compute df(x_arg, y_val)/dy @ v_val using jvp
    current_df_dy_v = jax.jvp(f_of_y_at_x_arg, (y_val,), (v_val,))[1]  # This is a p-vector

    # Compute the dot product with w_val
    return jnp.dot(w_val, current_df_dy_v)  # This is a scalar


# Step 3: Compute the gradient of the scalar function w.r.t. x
# jax.grad(func)(args) computes the gradient of func w.r.t. its first argument by default
# scalar_fn_of_x takes x_arg as its first argument
vJv_result = jax.grad(scalar_fn_of_x)(x_val)

# --- Print results ---
print(f"Shape of x: {x_val.shape}")
print(f"Shape of y: {y_val.shape}")
print(f"Shape of w: {w_val.shape}")
print(f"Shape of v: {v_val.shape}")
print(f"Shape of the result (w.T @ d^2f/dxdy @ v): {vJv_result.shape}")

# The result should be an n-dimensional vector
assert vJv_result.shape == (n,)

print("\nComputation successful.")
# print("Resulting vector:", vJv_result) # Uncomment to see the actual values

Shape of x: (5,)
Shape of y: (100,)
Shape of w: (10,)
Shape of v: (100,)
Shape of the result (w.T @ d^2f/dxdy @ v): (5,)

Computation successful.


In [3]:
import jax
import jax.numpy as jnp


# --- Define a small example function f: R^n x R^m -> R^p ---
# This function needs to be differentiable with respect to both x and y
def f(x, y):
    """
    Example function: f(x, y) = [x[0]*y[0]^2 + x[1]*y[1], sum(x)*sum(y), ...]
    Input: x (n,), y (m,)
    Output: vector (p,)
    """
    n_dim = x.shape[0]
    m_dim = y.shape[0]
    p_dim = 3  # Example output dimension

    # Ensure dimensions are compatible for this specific example f
    assert n_dim == x_val.shape[0]
    assert m_dim == y_val.shape[0]
    assert p_dim == w_val.shape[0]  # Output dim must match w

    # A simple, arbitrarily chosen differentiable function
    # f_0(x, y) = x[0] * y[0]**2 + x[1] * y[1] + jnp.sum(x) # assuming n >= 2, m >= 2
    # f_1(x, y) = jnp.sum(x) * jnp.sum(y) + x[0] # assuming n >= 1
    # f_2(x, y) = x[jnp.min(jnp.array([n-1, m-1]))] * y[jnp.min(jnp.array([n-1, m-1]))] # uses indexing

    # Let's define a simple one using common operations
    f0 = x[0] * jnp.sum(y**2) + x[1] * y[0]  # Requires n>=2, m>=1
    f1 = jnp.sum(x) * jnp.sum(y)
    f2 = x[-1] * y[-1] + jnp.dot(x, x)  # Requires n>=1, m>=1

    # Combine into a vector
    result = jnp.stack([f0, f1, f2])  # Shape (p,)
    return result


# --- Define small input dimensions ---
n = 2  # Dimension of x
m = 10  # Dimension of y
p = 3  # Dimension of the output of f (and w)

# --- Create some dummy data ---
key = jax.random.PRNGKey(0)
x_val = jax.random.normal(key, (n,))
y_val = jax.random.normal(key, (m,))
w_val = jax.random.normal(key, (p,))  # Needed for verification later
v_val = jax.random.normal(key, (m,))  # Needed for verification later

print(f"Input shapes: x={x_val.shape}, y={y_val.shape}")
print(f"Output dimension (p): {p}")

# --- Compute the full p x n x m tensor d^2f/dxdy ---

# Step 1: Compute the Jacobian of f w.r.t. x (p x n matrix)
jac_f_wrt_x_fn = jax.jacfwd(f, argnums=0)  # Returns a function (x, y) -> (p, n) array

# Step 2: Compute the Jacobian of the result from Step 1 w.r.t. y (p x n x m tensor)
# The function `jac_f_wrt_x_fn` takes (x, y). We differentiate w.r.t. its second argument (y).
full_hessian_mixed = jax.jacfwd(jac_f_wrt_x_fn, argnums=1)(x_val, y_val)

# --- Print the shape and the tensor ---
print(f"\nShape of the full mixed Hessian tensor d^2f/dxdy: {full_hessian_mixed.shape}")
print("Full mixed Hessian tensor d^2f/dxdy:")
print(full_hessian_mixed)

# Expected shape is (p, n, m)
assert full_hessian_mixed.shape == (p, n, m)


# --- Optional Verification: Calculate a single element using repeated grading ---
# This confirms the tensor elements correspond to d^2fi / dxj dyk

# Pick indices for verification
i0, j0, k0 = 1, 0, 2  # Example indices: 2nd output, derivative wrt x[0], then wrt y[2]

if i0 < p and j0 < n and k0 < m:
    # Step 1: Define a function for the i0-th output component
    def f_i0(x, y):
        return f(x, y)[i0]  # This is a scalar function

    # Step 2: Compute the gradient of f_i0 w.r.t. y
    # We need a function of y that returns the gradient
    def grad_fi0_wrt_y_fn(x_arg, y_arg):
        # jax.grad of a scalar function w.r.t. an argument
        return jax.grad(lambda y_inner: f_i0(x_arg, y_inner))(y_arg)  # Returns an (m,) vector

    # Step 3: Extract the k0-th component of the gradient from Step 2
    # This is the scalar function g(x, y) = d f_i0 / d y_k0
    def dfi0_dyk0_fn(x_arg, y_arg):
        return grad_fi0_wrt_y_fn(x_arg, y_arg)[k0]

    # Step 4: Compute the gradient of the scalar function dfi0_dyk0_fn w.r.t. x
    # This gives the vector d(dfi0/dyk0) / dx = d^2fi0 / (dx dyk0)
    def grad_dfi0dyk0_wrt_x_fn(x_arg, y_arg):
        return jax.grad(lambda x_inner: dfi0_dyk0_fn(x_inner, y_arg))(
            x_arg
        )  # Returns an (n,) vector

    # Step 5: Extract the j0-th component of the gradient from Step 4
    # This is the scalar d^2 fi0 / (dx_j0 dy_k0)
    element_val_repeated_grad = grad_dfi0dyk0_wrt_x_fn(x_val, y_val)[j0]

    # Get the same element from the full tensor
    element_val_full_tensor = full_hessian_mixed[i0, j0, k0]

    print(f"\nVerification for element d^2f_{i0}/(dx_{j0}dy_{k0}):")
    print(f"Value from full tensor: {element_val_full_tensor}")
    print(f"Value from repeated grading: {element_val_repeated_grad}")

    # Check if they are close (using a tolerance for floating point comparisons)
    assert jnp.isclose(element_val_full_tensor, element_val_repeated_grad), (
        f"Verification failed: {element_val_full_tensor} != {element_val_repeated_grad}"
    )
    print("Verification successful: Elements match.")
else:
    print("\nVerification indices out of bounds.")


# --- Demonstrate the vJv product calculation using the full tensor (INEFFICIENT) ---
# This is what the first method avoids calculating directly

# Reshape the full tensor for matrix multiplication
# The tensor has shape (p, n, m). We want to compute w.T @ Tensor @ v
# Conceptually, we can view this as (w.T @ T_matrix) @ v, where T_matrix is (p*n) x m
# Or w.T @ (Tensor @ v). Tensor @ v would be a (p, n) matrix (contracting over m)
# (Tensor @ v)_ij = sum_k Tensor_ijk * v_k = sum_k (d^2 fi / dxj dyk) * v_k
# This is the j-th component of the gradient w.r.t. x of (d fi / dy @ v)
# Let M_ij = sum_k (d^2 fi / dxj dyk) * v_k. This is a (p, n) matrix.
# We want w.T @ M = sum_i w_i * M_ij. This is an (n,) vector.
# (w.T @ M)_j = sum_i w_i * (sum_k d^2 fi / dxj dyk * v_k)
# This matches the definition sum_i sum_k w_i d^2 fi / dxj dyk * v_k

# Contract the full tensor with v along the last dimension (m)
# full_hessian_mixed is (p, n, m), v is (m,)
# Resulting matrix is (p, n)
intermediate_matrix = jnp.einsum("pnm, m -> pn", full_hessian_mixed, v_val)
# or using tensordot: jnp.tensordot(full_hessian_mixed, v_val, axes=((2,), (0,)))

# Contract the intermediate matrix with w along the first dimension (p)
# intermediate_matrix is (p, n), w_val is (p,)
# Resulting vector is (n,)
vJv_result_from_tensor = jnp.einsum("p, pn -> n", w_val, intermediate_matrix)
# or using tensordot: jnp.tensordot(w_val, intermediate_matrix, axes=((0,), (0,)))

print(f"\nvJv result calculated from full tensor (shape {vJv_result_from_tensor.shape}):")
print(vJv_result_from_tensor)


# --- Compare with the efficient vJv calculation method from the previous response ---


# Re-implement the efficient method for comparison
def efficient_vJv_method(x, y, w, v):
    # Compute JvP of f w.r.t. y in direction v
    def f_jvp_y_fn(x_arg, y_arg, v_vec):
        return jax.jvp(lambda y_inner: f(x_arg, y_inner), (y_arg,), (v_vec,))[1]

    # Define scalar function g(x) = w^T @ (df(x, y_const)/dy @ v_const)
    def scalar_fn_of_x(x_arg, y_const, w_const, v_const):
        df_dy_v = f_jvp_y_fn(x_arg, y_const, v_const)
        return jnp.dot(w_const, df_dy_v)

    # Compute the gradient of the scalar function w.r.t. x
    return jax.grad(scalar_fn_of_x)(x, y, w, v)


vJv_result_efficient = efficient_vJv_method(x_val, y_val, w_val, v_val)

print(f"\nvJv result calculated using the efficient method (shape {vJv_result_efficient.shape}):")
print(vJv_result_efficient)

# --- Final Comparison ---
print("\nComparing results from full tensor and efficient method:")
assert jnp.allclose(vJv_result_from_tensor, vJv_result_efficient), (
    "Results from full tensor and efficient method do not match!"
)
print("Results match! (within floating point tolerance)")

Input shapes: x=(2,), y=(10,)
Output dimension (p): 3

Shape of the full mixed Hessian tensor d^2f/dxdy: (3, 2, 10)
Full mixed Hessian tensor d^2f/dxdy:
[[[ 3.2452843  4.0505295 -0.8671889 -0.1572347  0.3521818 -1.9441785
   -0.9905975  0.9887572  1.3286986 -1.900327 ]
  [ 1.         0.         0.         0.         0.         0.
    0.         0.         0.         0.       ]]

 [[ 1.         1.         1.         1.         1.         1.
    1.         1.         1.         1.       ]
  [ 1.         1.         1.         1.         1.         1.
    1.         1.         1.         1.       ]]

 [[ 0.         0.         0.         0.         0.         0.
    0.         0.         0.         0.       ]
  [ 0.         0.         0.         0.         0.         0.
    0.         0.         0.         1.       ]]]

Verification for element d^2f_1/(dx_0dy_2):
Value from full tensor: 1.0
Value from repeated grading: 1.0
Verification successful: Elements match.

vJv result calculated from